# TotalEnergies India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** jobs.totalenergies.com (Avature)

**ATS:** Avature — pagination via jobOffset=0,20,40,...

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path
SCRIPTS_DIR = Path.home() / 'Job_Scrapers' / 'All_Scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
from scraper_utils import *
from bs4 import BeautifulSoup
from datetime import datetime
import requests
LOCATION_FILTER = 'India'
print('Imports loaded. Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-01 00:56:08


In [3]:
COMPANY = 'TotalEnergies'
OUTPUT_DIR = get_output_dir(COMPANY)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/TotalEnergies/Outputs/2026_04_01


In [4]:
print('=' * 60)
print('TOTALENERGIES INDIA JOB SCRAPER')
print('ATS: Avature (jobs.totalenergies.com)')
print('=' * 60)

BASE_URL = 'https://jobs.totalenergies.com'
SEARCH_URL = f'{BASE_URL}/en_US/careers/SearchJobs/'
PAGE_SIZE = 20
MAX_JOBS = 500

session = get_session()
session.headers.update({'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'})

totalenergies_jobs = []
seen_ids = set()

def parse_avature_page(soup, location_filter):
    jobs = []
    # Avature job cards are typically in <li> or <tr> elements with specific classes
    cards = (soup.select('li.job-result') or soup.select('[class*="job-result"]') or
             soup.select('[class*="search-result"]') or soup.select('article[class*="job"]'))
    if not cards:
        # Fallback: find all job links
        job_links = soup.select('a[href*="/en_US/careers/JobDetail"], a[href*="ViewJob"], a[href*="/job/"]')
        seen = set()
        for link in job_links:
            parent = link.find_parent(['li', 'tr', 'article', 'div'])
            if parent and id(parent) not in seen:
                cards.append(parent); seen.add(id(parent))
    for card in cards:
        title_el = (card.select_one('[class*="title"] a') or card.select_one('h2 a') or
                    card.select_one('h3 a') or card.select_one('a[href*="JobDetail"]') or
                    card.select_one('a[href*="ViewJob"]') or card.select_one('a'))
        title = title_el.get_text(strip=True) if title_el else ''
        if not is_valid_job_title(title): continue
        href = title_el.get('href', '') if title_el else ''
        job_url = href if href.startswith('http') else (BASE_URL + href if href else '')
        job_id = re.search(r'/(\d+)[/?]?', href)
        job_id = job_id.group(1) if job_id else str(abs(hash(title + job_url)))
        loc_el = card.select_one('[class*="location"]') or card.select_one('[class*="city"]')
        location_text = loc_el.get_text(strip=True) if loc_el else ''
        if location_filter and location_filter.lower() not in location_text.lower(): continue
        city = location_text.split(',')[0].strip() if location_text else 'India'
        dept_el = card.select_one('[class*="category"]') or card.select_one('[class*="department"]')
        dept = dept_el.get_text(strip=True) if dept_el else ''
        date_el = card.select_one('[class*="date"]') or card.select_one('time')
        raw_date = date_el.get_text(strip=True) if date_el else ''
        date_posted = datetime.now().strftime('%Y-%m-%d')
        for fmt in ('%d %b %Y', '%B %d, %Y', '%Y-%m-%d', '%m/%d/%Y'):
            try: date_posted = datetime.strptime(raw_date, fmt).strftime('%Y-%m-%d'); break
            except: continue
        jobs.append({'job_id': str(job_id), 'title': title, 'company_name': 'TotalEnergies',
                     'job_url': job_url, 'source_api_url': SEARCH_URL,
                     'business_unit': dept, 'raw_jd_text': card.get_text(' ', strip=True),
                     'location_city': city, 'location_country': 'India',
                     'industry': 'Energy / Oil & Gas', 'date_posted': date_posted,
                     'is_active': True, 'salary_currency': 'INR', 'source_platform': 'Avature'})
    return jobs

# Try requests-based scraping first (Avature renders server-side)
offset = 0
consecutive_empty = 0
while offset < MAX_JOBS:
    params = {'jobRecordsPerPage': PAGE_SIZE, 'jobOffset': offset}
    if LOCATION_FILTER:
        params['jobLocation'] = LOCATION_FILTER
    try:
        resp = session.get(SEARCH_URL, params=params, timeout=30)
        if resp.status_code != 200:
            print(f'  [ERROR] HTTP {resp.status_code} at offset={offset}'); break
        soup = BeautifulSoup(resp.text, 'lxml')
        page_jobs = parse_avature_page(soup, LOCATION_FILTER)
        new_jobs = [j for j in page_jobs if j['job_id'] not in seen_ids]
        for j in new_jobs: seen_ids.add(j['job_id'])
        totalenergies_jobs.extend(new_jobs)
        print(f'  offset={offset}: {len(new_jobs)} new jobs (total: {len(totalenergies_jobs)})')
        if not new_jobs:
            consecutive_empty += 1
            if consecutive_empty >= 2: print('  Stopping.'); break
        else: consecutive_empty = 0
        offset += PAGE_SIZE
        time.sleep(random.uniform(1.0, 2.0))
    except Exception as e:
        print(f'  [ERROR] {e}'); break

# Fallback: Selenium if requests returned too few results
if len(totalenergies_jobs) < 3:
    print('\n  Falling back to Selenium...')
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    driver = setup_selenium()
    try:
        offset = 0; consecutive_empty = 0
        while offset < MAX_JOBS:
            url = f'{SEARCH_URL}?jobRecordsPerPage={PAGE_SIZE}&jobOffset={offset}'
            if LOCATION_FILTER: url += f'&jobLocation={LOCATION_FILTER}'
            driver.get(url)
            try:
                WebDriverWait(driver, 20).until(EC.presence_of_element_located(
                    (By.CSS_SELECTOR, 'a[href*="JobDetail"],a[href*="ViewJob"],[class*="job-result"]')))
            except: time.sleep(5)
            soup = BeautifulSoup(driver.page_source, 'lxml')
            page_jobs = parse_avature_page(soup, LOCATION_FILTER)
            new_jobs = [j for j in page_jobs if j['job_id'] not in seen_ids]
            for j in new_jobs: seen_ids.add(j['job_id'])
            totalenergies_jobs.extend(new_jobs)
            print(f'  offset={offset}: {len(new_jobs)} new jobs (total: {len(totalenergies_jobs)})')
            if not new_jobs:
                consecutive_empty += 1
                if consecutive_empty >= 2: break
            else: consecutive_empty = 0
            offset += PAGE_SIZE; time.sleep(random.uniform(1.5, 2.5))
    except Exception as e:
        print(f'  [ERROR] {e}')
    finally:
        driver.quit()

print(f'\nTotal TotalEnergies India jobs scraped: {len(totalenergies_jobs)}')

TOTALENERGIES INDIA JOB SCRAPER
ATS: Avature (jobs.totalenergies.com)


  offset=0: 0 new jobs (total: 0)


  offset=20: 0 new jobs (total: 0)
  Stopping.

  Falling back to Selenium...


  offset=0: 0 new jobs (total: 0)


  offset=20: 0 new jobs (total: 0)

Total TotalEnergies India jobs scraped: 0


In [5]:
df_te = save_results(totalenergies_jobs, 'TotalEnergies', OUTPUT_DIR)
if df_te is not None:
    cols = ['title','location_city','seniority_level','business_unit','job_url']
    cols = [c for c in cols if c in df_te.columns]
    print(df_te[cols].head(10).to_string())

  [WARN] No jobs found for TotalEnergies
